In [1]:
import pandas as pd
from nltk.stem import WordNetLemmatizer
from pathlib import Path
import re
from tqdm import tqdm

In [2]:
MAX_LENGTH = 512
LABEL2VECTOR = {
    "NON-SUGGESTION": [1.0, 0.0],
    "SUGGESTION": [0.0, 1.0]
}
VECTOR2LABEL = {
    (1.0, 0.0): "NON-SUGGESTION",
    (0.0, 1.0): "SUGGESTION"
}

In [3]:
def process_text(text: str) -> str:
    lemmatizer = WordNetLemmatizer()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.lower().split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [4]:
def preproccess_for_model(examples, tokenizer):
    return tokenizer(
        examples["review_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

In [5]:
df = pd.read_csv("./data/Chile_all1_clean.csv", encoding="utf-8", sep=";")
df = df[df["language_detected_full"] == "en"]
df = df.head(10000)
df.shape

(10000, 7)

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# La ruta exacta que usaste para guardar
ruta_modelo = "../matusalem"

# 1. Cargar el tokenizador
# Esto lee los archivos vocab.txt, tokenizer_config.json, etc.
tokenizer = AutoTokenizer.from_pretrained(ruta_modelo)

# 2. Cargar el modelo
# Esto lee el archivo pytorch_model.bin (o model.safetensors) y el config.json
modelo = AutoModelForSequenceClassification.from_pretrained(ruta_modelo)

# (Opcional) Mover el modelo a la GPU si está disponible
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo.to(dispositivo)

# Poner en modo evaluación
modelo.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [7]:
textos = df["review_text"].fillna("").tolist()
batch_size = 32
predicciones = []

In [8]:
for i in tqdm(range(0, len(textos), batch_size), desc="Etiquetando dataset"):
    lote_textos = textos[i : i + batch_size]
    
    # Tokenizar el lote completo
    inputs = tokenizer(
        lote_textos, 
        return_tensors="pt", 
        truncation=True, 
        padding="max_length", 
        max_length=512 # Opcional: limitar el largo para no saturar memoria
    ).to(dispositivo)
    
    with torch.no_grad():
        outputs = modelo(**inputs)
        
    # Obtener las clases predichas de todo este lote
    logits = outputs.logits
    clases_lote = torch.argmax(logits, dim=-1).cpu().tolist()
    
    # Guardar los resultados
    predicciones.extend(clases_lote)

Etiquetando dataset: 100%|██████████| 313/313 [03:21<00:00,  1.55it/s]


In [9]:
df["etiqueta_matusalem"] = predicciones
df.to_csv("dataset_feedback_etiquetado.csv", index=False)

print("¡Etiquetado masivo completado con éxito!")

¡Etiquetado masivo completado con éxito!


In [16]:
df_olmo = pd.read_csv("./data/labeled/Chile_all1_clean_labeled.csv", encoding="utf-8", sep=";")
df_olmo = df_olmo[["review_text", "etiqueta"]]
df_olmo.shape

(1500, 2)

In [17]:
df_matusalem = pd.read_csv("dataset_feedback_etiquetado.csv")
df_matusalem = df_matusalem[["review_text", "etiqueta_matusalem"]].head(1500)
df_matusalem.shape

(1500, 2)

In [22]:
suma = 0

for i in range(df_olmo.shape[0]):
    if (df_olmo.iloc[i]['etiqueta'] == "NON-SUGGESTION" and df_matusalem.iloc[i]['etiqueta_matusalem'] == 1) or \
        (df_olmo.iloc[i]['etiqueta'] == "SUGGESTION" and df_matusalem.iloc[i]['etiqueta_matusalem'] == 0):
        print(f"COMENTARIO: {df_olmo.iloc[i]['review_text']}")
        print("ETIQUETA OLMO:", df_olmo.iloc[i]['etiqueta'])
        print("ETIQUETA MATUSALEM:", df_matusalem.iloc[i]['etiqueta_matusalem'])
        print("-" * 50)

        suma += 1

print(f"Total de discrepancias: {suma}")

COMENTARIO: The tour was fantastic and we learned a bit about Undurraga's connection to the Mapucho culture and how they support it with their TP brand. It would have been great had we been able to taste more premium wines but as close to Santiago, the tour is designed for maybe those newer to wines than those who are more vinofiles. Worth the trip and their gran reserva and icon wines are great - we bot those!
ETIQUETA OLMO: NON-SUGGESTION
ETIQUETA MATUSALEM: 1
--------------------------------------------------
COMENTARIO: My friends & I booked this tour in addition to another tour that we attended earlier that day. It was a bit last minute, so we were not expecting too much but it ended up being one of the most fantastic experiences that we had in Santiago. Our tour guide was absolutely flawless, his name is Chris I would highly recommend requesting him to be your guide. He is extremely informative, quickwitted and to be honest I am a wine snob, but he really knew his stuff and we we